# 02. pdf-inspector 응용 실습

목표: 실제 프로젝트에서 `pdf-inspector`를 붙인다고 가정하고, 분류 결과를 이용해 로컬 Markdown 추출과 OCR fallback을 나누는 코드를 작성합니다.

선택 설치:

```bash
pip install pdf-inspector
```

In [ ]:
from pathlib import Path


try:
    import pdf_inspector
except ImportError:
    pdf_inspector = None


def inspect_pdf_for_pipeline(path: str) -> dict:
    """PDF 하나를 처리 파이프라인 관점의 의사결정으로 변환합니다.

    패키지가 설치되어 있으면 실제 `detect_pdf`를 호출합니다.
    설치되어 있지 않으면 노트북을 읽는 학습자가 흐름을 이해할 수 있도록 안내 값을 반환합니다.
    """
    if pdf_inspector is None:
        return {
            "path": path,
            "status": "package_not_installed",
            "next_step": "pip install pdf-inspector 후 다시 실행",
        }

    result = pdf_inspector.detect_pdf(path)
    if result.pdf_type == "text_based" and not result.has_encoding_issues:
        route = "local_markdown"
    elif result.pages_needing_ocr:
        route = "page_level_ocr"
    else:
        route = "full_ocr_or_manual_review"

    return {
        "path": path,
        "pdf_type": result.pdf_type,
        "confidence": result.confidence,
        "pages_needing_ocr": result.pages_needing_ocr,
        "route": route,
    }


In [ ]:
sample_paths = [
    "data/report.pdf",
    "data/scanned_invoice.pdf",
    "data/mixed_contract.pdf",
]

for path in sample_paths:
    print(inspect_pdf_for_pipeline(path))


In [ ]:
def convert_to_markdown_if_possible(path: str) -> str:
    """텍스트 기반 PDF는 Markdown으로 변환하고, 아니면 fallback 메시지를 반환합니다."""
    if pdf_inspector is None:
        return "pdf-inspector가 설치되어 있지 않습니다. pip install pdf-inspector를 실행하세요."

    result = pdf_inspector.process_pdf(path)
    if result.markdown:
        return result.markdown

    return f"Markdown을 만들 수 없습니다. OCR 필요 페이지: {result.pages_needing_ocr}"


# 실제 PDF가 있을 때만 실행합니다.
candidate = Path("data/report.pdf")
if candidate.exists():
    print(convert_to_markdown_if_possible(str(candidate))[:1000])
else:
    print("data/report.pdf가 없어 변환 예제는 건너뜁니다.")


실무 포인트:

- 분류와 변환을 한 함수에 모두 숨기지 말고, 라우팅 결정을 로그로 남기면 장애 분석이 쉬워집니다.
- `pages_needing_ocr`를 저장하면 전체 문서를 OCR로 보내지 않아도 됩니다.
- 깨진 인코딩 신호가 있으면 텍스트가 있어도 OCR fallback을 고려해야 합니다.